In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Softmax Regression from Scratch

In that section we wrote linear regression as a handful of
plain functions, with tensors and automatic differentiation, so that no part
of training stayed hidden, and in that section we moved the
training loop into a `Trainer` so that we would never write it again. We
now write softmax regression the same way. What changes when the model
predicts a class instead of a number? Rather little, as it turns out.
Softmax regression differs from linear regression in three places. The model
computes one output per class instead of a single number, the softmax turns
these outputs, the logits, into probabilities, and the loss is the
cross-entropy instead of the squared error. Everything else carries over:
the `Trainer` runs the same loop with the same minibatch SGD update, and the
`FashionMNIST` class of that section delivers the images.

One piece of software is new, and every classifier in the book will use it.
A classifier is trained on its loss but judged by its accuracy, so we write
a base class `Classifier` that reports both. We write it, write the three
new pieces, and train the model on Fashion-MNIST. Writing the softmax from
its formula also exposes a numerical trap, which we demonstrate here and
leave to the next section to remove. Once the model is trained, the
interesting part begins. We look at what it gets wrong, at which garments it
takes for which, and, since the weights of a linear model on images are
themselves images, at what it has learned.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

## The Softmax

### Normalizing Each Row

We start with the softmax, the piece that turns the logits of the model into
probabilities. For a minibatch of $n$ examples the logits form a matrix
$\mathbf{O} = \mathbf{X}\mathbf{W} + \mathbf{b}$ with one row per example and
one column per class (that section). The softmax
acts on each row separately,

$$\mathrm{softmax}(\mathbf{O})_{ij} = \frac{\exp(O_{ij})}{\sum_{k=1}^q \exp(O_{ik})},$$

so entry $(i, j)$ of the result is the probability that the model assigns to
class $j$ for example $i$. Computing it takes three steps: exponentiate every
entry, sum each row, and divide each row by its sum. Only the second step
needs care with shapes. A sum with `axis=1` collapses the columns, and
`keepdims=True` keeps the collapsed axis with length 1, so the sums form a
column of shape $(n, 1)$ that broadcasts against the $n \times q$ matrix in
the division (that section).

In [ ]:
def softmax(O):
    O_exp = d2l.exp(O)
    return O_exp / d2l.reduce_sum(O_exp, 1, keepdims=True)  # Broadcast

We check it on two rows of logits. The first holds the logits
$(1.0, 2.2, 0.3)$ that that section computed for an image scored as
cat, chicken, and dog; the second holds three equal logits.

In [ ]:
O = d2l.tensor([[1.0, 2.2, 0.3], [0.0, 0.0, 0.0]])
Y_hat = softmax(O)
Y_hat, d2l.reduce_sum(Y_hat, 1)

The first row reproduces the probabilities $(0.21, 0.69, 0.10)$ computed by
hand, and equal logits give every class the probability $1/3$. Both rows sum
to 1, as probabilities must.

### Overflow

Unfortunately, the formula fails for large logits. For the logits
$(1000, 0, 0)$, $\exp(1000)$ exceeds the largest single-precision number
and becomes infinite, in the numerator of the first entry and in the row
sum alike, and $\infty / \infty$ is `NaN`, although the correct answer is
$(1, 0, 0)$ up to rounding. The remedy is the invariance of
that section: the softmax ignores a constant added to all logits
of a row, so we may subtract the largest logit of each row first. Every
exponent is then at most 0, and nothing overflows. The framework's built-in
softmax shifts the logits in the same way. We apply the direct formula, the
shifted formula, and the built-in softmax to the same row:

In [ ]:
O = d2l.tensor([[1000.0, 0.0, 0.0]])
O_max = tf.reduce_max(O, axis=1, keepdims=True)
softmax(O), softmax(O - O_max), tf.nn.softmax(O, axis=1)

The direct formula returns `NaN` in the first entry; the shifted formula and
the built-in softmax return $(1, 0, 0)$. A `NaN` does not stay where it
arises. The loss of the minibatch becomes `NaN`, then its gradient, and after
the update the parameters, and the model is lost. Our model is in no danger,
since its inputs are pixel values between 0 and 1 and its weights stay
small, so we keep the direct formula, which shows the three steps plainly.
Nor is the shift the whole answer. It saves the softmax but not the loss,
which takes the logarithm of the result, and the logarithm of a probability
that has underflowed to 0 is $-\infty$. that section
removes both problems at once, by computing the logarithms of the
probabilities directly from the logits.

## Loss and Accuracy

Training a classifier lowers its cross-entropy loss. The user of a
classifier has little interest in the loss and asks a different question:
how often is the decision right? Every classifier in this book therefore
reports two numbers on validation data: the loss, because training lowers
it, and the accuracy, because users ask for it. Both go into a base class
`Classifier` that extends the `Module` class of that section,
so that each model supplies only its network and its loss.

### Two Uses of the Logits

A classifier for $q$ classes computes a vector of logits
$\mathbf{o} \in \mathbb{R}^q$ for each example and predicts the class with
the largest logit. Since the softmax preserves the order of the logits, this
is also the class with the largest probability. Let
$y^{(i)} \in \{1, \ldots, q\}$ be the class of example $i$. The accuracy on
$n$ examples is the fraction of examples whose largest logit belongs to the
true class,

$$\textrm{accuracy} = \frac{1}{n} \sum_{i=1}^n \mathbf{1}\Big(\mathop{\mathrm{argmax}}_j o^{(i)}_j = y^{(i)}\Big),$$

where $\mathbf{1}(\cdot)$ is 1 if its argument is true and 0 otherwise: a
right prediction counts 1, a wrong one 0, and the accuracy is their average.
A classifier that labels 9,000 of the 10,000 Fashion-MNIST validation
images correctly has accuracy 0.9; its error rate, one minus the accuracy,
is 0.1.

We report the accuracy but cannot train on it, since it changes only in
jumps as the parameters change (that section). Gradient descent
needs the smooth cross-entropy loss instead. Luckily, both numbers come from
the same vector of logits, as the figure shows.

![One logit vector, two uses. Training (top) turns the logits into probabilities with the softmax and reads off the cross-entropy loss of the true class, which changes smoothly with the parameters. Evaluation (bottom) takes the class with the largest logit and compares it with the label, which counts toward the accuracy and changes in steps. The logits $(1.0, 2.2, 0.3)$ score the classes cat, chicken, and dog for an image of a chicken.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-clf-loss-accuracy.svg)

It is tempting to assume that the two numbers at least move together. They
need not. A model that becomes confidently wrong on a few validation
examples pays a large loss on each of them, so its validation loss can rise
while its accuracy still improves. This is why we report both.

### The `Classifier` Class

`Classifier` has a single job: on each validation minibatch, compute the
loss and the accuracy. To this end its `validation_step` replaces that of
`Module`, which returns only the loss, by one that returns a dictionary with
both metrics. The `Trainer` of that section does the rest. It
averages each metric that a step returns over the validation set, weighting
every minibatch by its number of examples, so that the reported loss and
accuracy are exact means over the validation set. After each epoch it
appends the two means to `trainer.history['val_loss']` and
`trainer.history['val_acc']`.

As in `Module`, the validation step calls the model with `training=False`.

In [ ]:
class Classifier(d2l.Module):
    """The base class of classification models."""
    def validation_step(self, batch):
        Y_hat = self(*batch[:-1], training=False)
        return {'loss': self.loss(Y_hat, batch[-1]),
                'acc': self.accuracy(Y_hat, batch[-1])}

This is all there is to the validation step. What the class does not
contain is a network or a loss. These belong to its subclasses, and the rest
of this section writes both for softmax regression.

### Computing Accuracy

The method `accuracy` evaluates the equation on a minibatch. `Y_hat`
holds the outputs of the model, one row per example (more leading axes are
flattened first); logits and probabilities have their largest entry in the
same place. `argmax` along the class axis gives the predicted classes. We cast
them to the data type of the labels `Y`, a formality that is needed because
some frameworks compare only values of the same type. The comparison yields
1 for each correct prediction and 0 for each wrong one, and the mean of
these values is the accuracy. With `averaged=False` the method returns the vector
of 0s and 1s itself. This is the form to ask for when we want to look at
the examples that the model gets wrong.

In [ ]:
@d2l.add_to_class(Classifier)
def accuracy(self, Y_hat, Y, averaged=True):
    """Compute the fraction of correct predictions."""
    Y_hat = d2l.reshape(Y_hat, (-1, Y_hat.shape[-1]))
    preds = d2l.astype(d2l.argmax(Y_hat, axis=1), Y.dtype)
    compare = d2l.astype(preds == d2l.reshape(Y, (-1,)), d2l.float32)
    return d2l.reduce_mean(compare) if averaged else compare

### The Default Optimizer

Unless a model asks for something else, it trains with minibatch stochastic
gradient descent at its learning rate `self.lr`, as the linear regression of
that section did. The `Trainer` obtains the optimizer once, at
the start of training, by calling the model's method `configure_optimizers`.
Nothing in this default is specific to classification, so we install it on
`Module` rather than on `Classifier`, and regression models inherit it as
well. Later chapters override it to train with other optimizers.

In [ ]:
@d2l.add_to_class(d2l.Module)
def configure_optimizers(self):
    return tf.keras.optimizers.SGD(float(self.lr))

## The Model and the Loss

### The Parameters

Each $28 \times 28$ image becomes a vector of $d = 784$ pixel values, and
there are $q = 10$ classes. The model therefore has a weight matrix
$\mathbf{W} \in \mathbb{R}^{784 \times 10}$, one column per class, and a bias
vector $\mathbf{b} \in \mathbb{R}^{10}$, together 7,850 parameters. As in
that section, we draw the weights from a normal distribution
with mean 0 and standard deviation 0.01 and set the biases to 0. Such small
weights give all classes nearly equal logits, so the model starts with
predictions close to uniform, $\hat{y}_j \approx 0.1$, and a loss close to
$\log 10 \approx 2.30$. The number is worth remembering: a first minibatch
that reports a loss far from 2.30 usually points to a bug.

We register $\mathbf{W}$ and $\mathbf{b}$ as parameters of the module with
the framework's parameter type, as in that section. The
default optimizer that `Classifier` inherits from `Module`, minibatch SGD at
the learning rate `lr`, finds registered parameters by itself, so this model needs no `configure_optimizers` of its
own.

`add_weight` creates each variable and registers it with the Keras model, so
`model.trainable_variables` lists `W` and `b`.

In [ ]:
class SoftmaxRegressionScratch(Classifier):
    """Softmax regression implemented from scratch."""
    def __init__(self, num_inputs, num_outputs, lr, std=0.01):
        super().__init__()
        self.save_hyperparameters()
        normal = tf.keras.initializers.RandomNormal(mean=0, stddev=std)
        self.W = self.add_weight(name='W', shape=(num_inputs, num_outputs),
                                 initializer=normal)
        self.b = self.add_weight(name='b', shape=(num_outputs,),
                                 initializer='zeros')

### The Forward Pass

A minibatch of $n$ images arrives as a four-dimensional tensor
(that section). The method `forward` reshapes it into the
$n \times 784$ matrix $\mathbf{X}$, one flattened image per row, computes the
logits $\mathbf{O} = \mathbf{X}\mathbf{W} + \mathbf{b}$, with the bias added
to every row by broadcasting, and returns their softmax. Flattening discards
which pixels are neighbors. The model cannot use that information; the
convolutional networks of that section do.

The minibatch has shape $(n, 28, 28, 1)$, channels last.

In [ ]:
@d2l.add_to_class(SoftmaxRegressionScratch)
def forward(self, X):
    X = d2l.reshape(X, (-1, self.num_inputs))  # One flattened image per row
    return softmax(d2l.matmul(X, self.W) + self.b)

### The Cross-Entropy Loss

The loss of one example is the negative logarithm of the probability that
the model assigns to its true class, $l = -\log \hat{y}_y$
(that section). For a minibatch we need, from each row of
$\hat{\mathbf{Y}}$, the entry in the column of that example's label. The
indexing of that section does this without a loop: the row
positions together with the vector of labels pick one entry per row. In code
the labels are class indices counted from 0. Here are two examples with
three classes and labels 0 and 2:

As in that section, `tf.gather` with `batch_dims=1` picks, in each
row, the entry at that row's label.

In [ ]:
y_hat = tf.constant([[0.1, 0.3, 0.6], [0.3, 0.2, 0.5]])
y = tf.constant([0, 2])
tf.gather(y_hat, y, batch_dims=1)

The first example gives its true class the probability 0.1, the second
0.5. The function `cross_entropy` averages the negative logarithms of these
probabilities over the minibatch. Its one line that is not part of the
definition raises each probability to at least $10^{-12}$ before the
logarithm, a crude guard that turns the infinite loss of a probability that
has underflowed to 0 into $27.6$ and, as the comment warns, into a zero
gradient; the next section replaces it by a loss that never forms the
probability.

In [ ]:
def cross_entropy(y_hat, y):
    p = tf.gather(y_hat, y, batch_dims=1)
    # Tiny clip to keep log finite when softmax outputs underflow to 0.
    return -tf.reduce_mean(tf.math.log(tf.maximum(p, 1e-12)))

cross_entropy(y_hat, y)

The loss is the mean of $-\log 0.1 \approx 2.30$ and
$-\log 0.5 \approx 0.69$, about 1.50. It becomes the loss of our model:

In [ ]:
@d2l.add_to_class(SoftmaxRegressionScratch)
def loss(self, y_hat, y):
    return cross_entropy(y_hat, y)

## Training and Evaluation

### The Training Run

With the parameters, the forward pass, and the loss in place, the `Trainer`
can fit the model to Fashion-MNIST. We train for 10 epochs with minibatches
of 256 images at learning rate 0.1, which makes 235 steps per epoch, and
the `Trainer` reports the loss and accuracy on the 10,000 validation images
after every epoch. None of the three numbers we just chose is learned, yet
each influences the result. Choosing them by their validation accuracy is a
form of model selection, and that section
asks what such choices do to the trustworthiness of the score.

In [ ]:
data = d2l.FashionMNIST(batch_size=256)
model = SoftmaxRegressionScratch(num_inputs=784, num_outputs=10, lr=0.1)
trainer = d2l.Trainer(max_epochs=10)
trainer.fit(model, data)

The training loss falls quickly during the first epoch and slowly after it.
The validation accuracy is between 75% and 80% after one epoch and about 83%
after ten. It does not rise steadily: from one epoch to the next it can
drop by a point or two, occasionally more, since minibatch SGD keeps moving
the parameters. The training loss ends near 0.45 and the validation loss
only a few hundredths above it.

Validation metrics are also needed outside `fit`, for instance to evaluate
a trained model once on a test set. We add the method `evaluate` to the
trainer. It returns the means of the metrics of `model` over
`data.val_dataloader()`, without drawing a single training minibatch, by
calling the same `validate` that `fit` calls after every epoch
(that section). With `train=True` it reads the training data
instead, still without changing the parameters.

In [ ]:
@d2l.add_to_class(d2l.Trainer)
def evaluate(self, model, data, train=False):
    """Metrics of `model` on the validation data of `data` (on its training
    data if `train`), averaged over examples."""
    self.train_dataloader = None
    self.val_dataloader = (data.train_dataloader() if train
                           else data.val_dataloader())
    self.prepare_model(model)
    return self.validate()

Here the metrics are means over all 10,000 validation images. After training
they equal the last point of each validation curve, which
`trainer.history['val_loss'][-1]` and `trainer.history['val_acc'][-1]`
hold.

In [ ]:
trainer.evaluate(model, data)

The model labels about 83% of the validation images correctly, and the
number should not be quoted more precisely than that. Repeated runs land
anywhere between 82% and 84%, since the random initialization and the order
of the minibatches move the result by about a point
(that section explains why we report such ranges).

How does this compare with the training images? The training curve above
averages the loss over each epoch while the parameters still change, so we
evaluate the trained model on all 60,000 training images.

In [ ]:
trainer.evaluate(model, data, train=True)

The training accuracy is about 84%, one or two points above the validation
accuracy: the model does almost as well on images it has not seen as on the
images it was fitted to. A gap this small is no accident. A linear model
cannot fit arbitrary labels of 60,000 images, so it can memorize little of
its training data, and that section says
how large the gap could have been.

### Misclassified Images

The accuracy tells us how often the model errs, not on which images. We take
the first minibatch of 256 validation images, predict the class of each, and
count the mistakes.

In [ ]:
X, y = next(iter(data.val_dataloader()))
preds = d2l.astype(d2l.argmax(model(X), axis=1), y.dtype)
wrong = preds != y
print(f'{int(d2l.reduce_sum(d2l.astype(wrong, d2l.int32)))} of {len(y)} wrong')

We show the first eight misclassified images. Each title gives the true
class above the predicted one.

In [ ]:
labels = [f'{true}\n→ {pred}' for true, pred in zip(
    data.text_labels(y[wrong]), data.text_labels(preds[wrong]))]
data.visualize([X[wrong], y[wrong]], labels=labels)

Between 30 and 45 of the 256 images are misclassified, in line with an
error rate of about 17%. The mistakes are not wild ones. Most of them take
one top for another or one shoe for another. In fairness to
the model, some of these images are hard to label at $28 \times 28$ pixels
even for a person.

### The Confusion Matrix

Eight images are anecdotes. A *confusion matrix* $\mathbf{N}$ counts all
the mistakes. For $q$ classes it has $q \times q$ entries. Let
$\hat{c}^{(m)} = \operatorname{argmax}_j o^{(m)}_j$ be the predicted class
of example $m$, the class with the largest logit; we keep
$\hat{\mathbf{y}}$ for the predicted probabilities. Then

$$N_{ij} = \sum_{m=1}^n \mathbf{1}\big(\hat{c}^{(m)} = i\big)\, \mathbf{1}\big(y^{(m)} = j\big)$$

counts the examples of class $j$ predicted as class $i$: rows are predicted
classes and columns true classes. The diagonal holds the correct
predictions, so the accuracy is the sum of the diagonal divided by $n$, and
each off-diagonal entry counts one kind of mistake. Libraries do not agree
on the orientation; scikit-learn's `confusion_matrix`, for one, puts the
true classes in the rows. We put them in the columns for a reason that
that section will use. Once each column
is divided by its sum, the matrix acts on a vector of class proportions and
returns the frequencies of the predictions, $\mathbf{C}\mathbf{p} = \boldsymbol{\mu}$, an equation that can be solved for $\mathbf{p}$.

`tf.math.confusion_matrix` also puts the true classes in the rows and the
predicted classes in the columns, the transpose of $\mathbf{N}$.

How do we count 100 kinds of pairs without a loop over examples? One-hot
vectors (that section) do it. Let the rows of
$\mathbf{P} \in \{0, 1\}^{n \times q}$ be the one-hot encodings of the
predicted classes of a minibatch and the rows of
$\mathbf{Y} \in \{0, 1\}^{n \times q}$ those of the labels. Then

$$(\mathbf{P}^\top \mathbf{Y})_{ij} = \sum_{m=1}^n P_{mi} Y_{mj},$$

and a term is 1 exactly when example $m$ is predicted as $i$ and labeled
$j$, as in the equation. One matrix product thus counts all pairs of a minibatch, and summing
the products over the validation minibatches gives $\mathbf{N}$. The function
`one_hot` builds the one-hot rows by comparing a column of class indices with
the row $(0, 1, \ldots, q-1)$, which broadcasting expands to an $n \times q$
matrix of comparisons.

In [ ]:
def one_hot(y, q):
    """Rows of the q x q identity matrix, one per class index in y."""
    return d2l.astype(d2l.reshape(y, (-1, 1)) == d2l.arange(q, dtype=y.dtype),
                      d2l.float32)

N = d2l.zeros((10, 10))
for X, y in data.val_dataloader():
    P = one_hot(d2l.argmax(model(X), axis=1), 10)
    N += d2l.matmul(d2l.transpose(P), one_hot(y, 10))
int(d2l.reduce_sum(N))

The counts add up to the 10,000 validation images, so no pair was lost.
Counts depend on how many examples each class happens to have. To compare
classes, we divide each column of $\mathbf{N}$ by its sum. This gives the
matrix $\mathbf{C}$ with $C_{ij} = N_{ij} / \sum_{i'} N_{i'j}$, the
fraction of the examples of class $j$ that are predicted as class $i$, an
estimate of $P(\hat{c} = i \mid y = j)$. Column $j$ shows where the
examples of class $j$ end up, and the diagonal holds, for each class, the
fraction of its examples that the model recognizes. The display utility
`d2l.show_heatmaps` draws $\mathbf{C}$ as a grid of colored cells.

In [ ]:
C = N / d2l.reduce_sum(N, 0, keepdims=True)  # Each column sums to 1
names = data.text_labels(range(10))
d2l.show_heatmaps(d2l.reshape(C, (1, 1, 10, 10)), xlabel='true class',
                  ylabel='predicted class', figsize=(4, 4), cmap='Blues',
                  xticklabels=names, yticklabels=names)

The largest entries off the diagonal name the most frequent mistakes:

In [ ]:
mistakes = d2l.numpy(N) * (1 - np.eye(10))  # Zero the diagonal
for k in np.argsort(mistakes, axis=None)[::-1][:5]:
    i, j = divmod(int(k), 10)
    print(f'{names[j]} taken for {names[i]}: {int(mistakes[i, j])}')

The diagonal shows how often each class is recognized. Trousers, bags, and
the three kinds of shoes are recognized about 90% of the time or more, and
shirts only about half the time, between 40% and 60% depending on the run.
The errors are far from evenly spread. They form the two groups that
that section led us to expect. The four tops (T-shirts,
pullovers, coats, and shirts) trade most of their errors among themselves:
in every run, the five largest entries off the diagonal are pairs of tops,
and the largest of them, a pullover taken for a coat or for a shirt, or a
coat taken for a pullover, holds between a sixth and a third of its class.
Sandals, sneakers, and ankle boots form a second, smaller group, whose
errors almost all stay among the three. The four tops have similar outlines
and differ in details such as collars, buttons, and the length of the
sleeves, which change few pixels. A shirt resembles each of the other three
in some of these details, so it collects errors from all of them.

### The Learned Templates

Why these mistakes and not others? The weights tell. The score of class
$j$ is the inner product of the flattened image with column $j$ of
$\mathbf{W}$, plus a bias. Reshaped to $28 \times 28$ pixels, the column is
itself an image, a *template* for class $j$: a bright pixel where the
template is red raises the score of the class, and a bright pixel where it
is blue lowers it. We draw the ten templates on one color scale, centered
at 0.

In [ ]:
W = d2l.numpy(model.W)
templates = W.T.reshape(10, 28, 28)  # Column j of W as an image
v = np.abs(templates).max()
d2l.show_images(templates, 2, 5, titles=names, cmap='RdBu_r', vmin=-v, vmax=v);

Each template is a blurred garment, and the ten of them are the whole
model: to classify an image is to ask which template it overlaps most.
Templates of the same form can also be counted instead of trained, from how
often each pixel is inked in each class; that is the naive Bayes classifier
of that section. The
templates of trousers, bags, and shoes have outlines that no other class
shares; the trouser template is blue between the legs, where a dress is
bright and a trouser is dark. The four tops share most of their outline,
and their templates differ mainly at the neckline and the sleeves: a few
pixels, and the same few in which the confusion matrix says they are
confused.

More training will not remove these errors. Every class score is a weighted
sum of pixels, so the boundaries between the predicted classes are
hyperplanes in the space of images, whatever the weights, and a collar is
too few pixels to tip such a sum reliably. Models that first compute
features from groups of pixels, such as the multilayer perceptrons of
that section, separate the classes better. Before we go
there, one piece of this implementation still needs repair: a softmax that
returns `NaN`, and a loss that can only hide it.

## Summary

Softmax regression from scratch adds three components to the training code
of linear regression: a weight matrix
$\mathbf{W} \in \mathbb{R}^{784 \times 10}$ and a bias
$\mathbf{b} \in \mathbb{R}^{10}$ registered as parameters, a forward pass
that flattens the images and applies the softmax to each row of
$\mathbf{O} = \mathbf{X}\mathbf{W} + \mathbf{b}$, and the cross-entropy loss,
which picks the probability of the true class in each row by indexing. The
base class `Classifier` reports, on every validation minibatch, the loss
that training lowers and the accuracy that users ask for, which changes in
steps and cannot be trained on. Ten epochs of minibatch SGD reach about 83%
validation accuracy and about 84% on the training images. The confusion
matrix, a sum of products of one-hot matrices, shows that most errors trade
one top for another, and the learned weights, viewed as ten templates, show
why: the tops differ in few pixels. The softmax written from its formula
returns `NaN` when a large logit overflows the exponential, and the floor in
the loss only hides the matching failure of the logarithm; the next section
computes the loss from the logits instead.

## Exercises

1. [code] **Underflow.** The `softmax` of this section returns `NaN` on
   the logits $(1000, 0, 0)$, and subtracting the largest logit of the row
   fixes it.
    1. Show that `softmax` also returns `NaN` on the row
       $(-1000, -1000, -1000)$, and identify the operation that produces
       the `NaN` in this case. Check that the shifted version returns
       $(1/3, 1/3, 1/3)$.
    1. Take the logits $(0, -200, 0)$ and the label `y = 1`, the class of
       the logit $-200$. Compute the shifted softmax and `cross_entropy` of
       the result. Then compute the
       gradient of this loss with respect to the logits by automatic
       differentiation, and compare it with the gradient
       $\hat{\mathbf{y}} - \mathbf{y}$ derived in that section.
       Explain the difference, and what it means for training on an
       example that the model gets badly wrong.

1. [code] **Cross-entropy from one-hot labels.** The `cross_entropy` of this
   section selects the correct-class probability by indexing. Implement an
   alternative that follows the definition $-\sum_j y_j \log \hat{y}_j$
   literally, by converting `y` to a one-hot matrix and taking row-wise
   inner products.
    1. Use it as the `loss` of `SoftmaxRegressionScratch` and confirm that
       training is unchanged.
    1. Time both versions on random probability matrices with $256$ rows
       and $q \in \{10, 10^3, 10^5\}$ columns, and state how the memory
       that each version allocates grows with $q$.
    1. Describe a setting in which the inner-product form applies but the
       indexing form does not.

1. **Large vocabularies.** Suppose that softmax regression predicts the
   next word from a feature vector of dimension $d = 1024$. The vocabulary
   has $V = 10^6$ words, and each minibatch holds $n = 256$ examples.
   List the operations of `forward` and `cross_entropy` whose cost grows
   with $V$. Compute the multiply-adds of the matrix product, the float32
   memory of `W`, and the float32 memory of the $n \times V$ matrix of
   probabilities, and compare each number with the Fashion-MNIST model of
   this section.

1. [code] **Hyperparameter sweep.** Retrain `SoftmaxRegressionScratch` with
   `d2l.Trainer(max_epochs=10)`, varying one hyperparameter at a time, and
   read the final losses from `trainer.history`.
    1. Plot the final validation loss against the learning rate over
       $\{0.01, 0.03, 0.1, 0.3, 1\}$.
    1. Plot the final training and validation losses against the minibatch
       size over $\{8, 32, 256, 2048\}$ at learning rate $0.1$.
    1. Compute the number of parameter updates that ten epochs over the
       60,000 training images take at each minibatch size. Use these counts
       to explain the trend in the previous plot, and predict how the
       comparison changes when every run takes the same number of updates.

1. [code] **Templates and class means.** The templates are what minibatch
   SGD made of the pixels it saw.
    1. Train `SoftmaxRegressionScratch` again with `std=0`, so that every
       weight starts at 0 and every class starts with the same logit.
       Explain why training works all the same, and compare the accuracy
       and the templates with those of the text.
    1. Show that after one SGD step from zero weights, column $j$ of
       $\mathbf{W}$ is proportional to
       $\sum_i \big(y^{(i)}_j - \tfrac{1}{10}\big)\, \mathbf{x}^{(i)}$ over
       the first minibatch: the sum of the images of class $j$ minus a
       tenth of the sum of all images.
    1. Compute the mean image of each class over the training set and
       compare it with the trained template of that class, by eye and by
       the correlation of their pixels. Where do they agree, and where does
       a template depart from the mean? Explain why a template that merely
       copied the class mean would confuse the tops even more.

[Discussions](https://d2l.discourse.group/t/225)